# SRMP Session 4: The planet zoo, in Python

Last week you made this plot by clicking on the NASA Exoplanet Archive website. Today you make the same plot yourself in code, and you can see what the exoplanet archive was doing behind the scenes.

**How to use this notebook:** read each cell, run it with Shift+Enter, and look at what comes out. Cells marked **YOUR TURN** have something for you to fill in or fix.

## 1. Load the data

The website plotting tool was reading a big table. Here is the same table, saved as a CSV file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# This is the Planetary Systems Composite table from the NASA Exoplanet Archive,
#  One row per planet.
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+pl_name,hostname,discoverymethod,disc_year,pl_orbper,pl_orbsmax,pl_bmassj,pl_bmasse,pl_rade,pl_radj,pl_eqt,st_mass,st_met,sy_dist+from+pscomppars&format=csv"
df = pd.read_csv(url)

print("Number of planets:", len(df))
df.head()

## 2. What's in the table?

Every column has a short code name. Here is the decoder ring:

| column | meaning | unit |
|---|---|---|
| `pl_name` | planet name | |
| `discoverymethod` | how it was found | |
| `disc_year` | year it was announced | |
| `pl_orbper` | orbital period | days |
| `pl_orbsmax` | semi-major axis | AU |
| `pl_bmassj` | planet mass (or mass × sin i) | Jupiter masses |
| `pl_bmasse` | same thing | Earth masses |
| `pl_radj` | planet radius | Jupiter radii |
| `pl_rade` | planet radius | Earth radii |
| `pl_eqt` | equilibrium temperature | K |
| `st_mass` | star mass | Sun masses |
| `st_met` | star metallicity [Fe/H] | dex |
| `sy_dist` | distance to the system | parsecs |

In [ ]:
# How many planets were found by each method?
df["discoverymethod"].value_counts()

**YOUR TURN:** Change the cell above to count planets by `disc_year` instead. Which year had the most discoveries? (Hint: `.sort_index()` at the end puts them in year order.)

In [ ]:
# your code here


## 3. Rebuild the website plot

Mass vs period, log axes, colored by discovery method. If this matches what you made last week, you've learned matplotlib.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for method, group in df.groupby("discoverymethod"):
    ax.scatter(group["pl_orbper"], group["pl_bmassj"], s=12, label=f"{method} ({len(group)})")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Orbital period [days]")
ax.set_ylabel("Planet mass or mass·sin(i) [Jupiter masses]")
ax.set_title("Every confirmed exoplanet")
ax.legend(fontsize=8, markerscale=2)
plt.show()

**Talk about it:**
- Why are the transit planets (the big purple-ish cloud) all on the left?
- Why is the bottom right corner empty? Is it empty in reality, or just in our data?
- Why does the y axis say "mass·sin(i)"?

## 4. Add the Solar System

None of our own planets are in the table, since it's a list of *exo*planets. Let's add them by hand so we can see where home sits.

In [ ]:
solar_system = {
    # name: (period in days, mass in Jupiter masses, semi-major axis in AU)
    "Mercury": (88,    0.000174, 0.39),
    "Venus":   (225,   0.00256,  0.72),
    "Earth":   (365,   0.00315,  1.00),
    "Mars":    (687,   0.000338, 1.52),
    "Jupiter": (4333,  1.0,      5.20),
    "Saturn":  (10759, 0.299,    9.58),
    "Uranus":  (30687, 0.0457,   19.2),
    "Neptune": (60190, 0.0540,   30.1),
}

fig, ax = plt.subplots(figsize=(10, 7))

# all exoplanets in gray this time, so the Solar System stands out
ax.scatter(df["pl_orbper"], df["pl_bmassj"], s=8, color="lightgray", label="Exoplanets")

for name, (P, M, a) in solar_system.items():
    ax.scatter(P, M, s=80, marker="*", color="black", zorder=5)
    ax.annotate(name, (P, M), xytext=(5, 5), textcoords="offset points", fontsize=9)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Orbital period [days]")
ax.set_ylabel("Planet mass [Jupiter masses]")
ax.set_title("Where does the Solar System sit?")
plt.show()

**Talk about it:** Which of our planets sit in a crowded part of the plot, and which sit alone? If an alien astronomer 50 light years away used our current methods, which of our planets would they have found?

## 5. Draw the Gaia box

Lammers & Winn (2025) predict that Gaia DR4 will find about 7,500 planets, mostly **3 to 13 Jupiter masses** at **2 to 5 AU**.

Our x axis is period, not AU. Kepler's third law converts between them. For a star like the Sun,

$$P_{\rm years} = \sqrt{a_{\rm AU}^3}$$

**YOUR TURN:** fill in the two missing numbers. What is the period in days for a = 2 AU and for a = 5 AU?

In [ ]:
def au_to_days(a):
    # Kepler's third law for a Sun-like star
    P_years = np.sqrt(a**3)
    return P_years * 365.25

P_low  = au_to_days(2)   # <-- what does this come out to?
P_high = au_to_days(5)

print(f"2 AU -> {P_low:.0f} days")
print(f"5 AU -> {P_high:.0f} days")

In [ ]:
M_low, M_high = 3, 13   # Jupiter masses

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(df["pl_orbper"], df["pl_bmassj"], s=8, color="lightgray")

for name, (P, M, a) in solar_system.items():
    ax.scatter(P, M, s=80, marker="*", color="black", zorder=5)

# the Gaia box
import matplotlib.patches as patches
box = patches.Rectangle((P_low, M_low), P_high - P_low, M_high - M_low,
                        fill=False, edgecolor="red", linewidth=2, label="Where Gaia DR4 planets should land")
ax.add_patch(box)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Orbital period [days]")
ax.set_ylabel("Planet mass [Jupiter masses]")
ax.legend()
plt.show()

## 6. Count what's already in the box

This is the number we write down and come back to in the spring. A *boolean mask* is a list of True/False, one per row, that picks out the rows we want.

In [ ]:
in_box = (df["pl_orbper"] > P_low) & (df["pl_orbper"] < P_high) & \
         (df["pl_bmassj"] > M_low) & (df["pl_bmassj"] < M_high)

print("Planets already known inside the Gaia box:", in_box.sum())
print("Lammers & Winn predict Gaia DR4 will add about:", 7500)
print()
print("Which methods found the ones we already have?")
print(df[in_box]["discoverymethod"].value_counts())

**Write this down with today's date:** _____ planets known in the box, found mostly by _____.

**Talk about it:** Why has radial velocity found some planets in this box but transits almost none? (Think about how often a 3 to 11 year orbit repeats.)

## 7. YOUR TURN: a different plot

Pick one of these and make it. Copy the plotting code from section 3 and change the column names.

1. **Radius vs period** (`pl_rade` vs `pl_orbper`), colored by method. Look for the thin empty gap near 1.8 Earth radii.
2. **Mass vs radius** (`pl_bmasse` vs `pl_rade`), colored by `pl_eqt`. Try `ax.scatter(..., c=group["pl_eqt"], cmap="plasma")` with no groupby. Rocky, icy and gassy planets separate into bands.
3. **Metallicity vs mass** (`st_met` vs `pl_bmassj`). Do metal-rich stars host more giants?

Then write three sentences: what the axes are, one pattern you see, and one reason it might be a selection effect rather than nature.

In [ ]:
# your plot here


## 8. Stretch: the discovery waves

A histogram of discovery year. Can you spot Kepler (launched 2009, main results 2014 to 2016) and TESS (launched 2018)? DR4 will be the next wave.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["disc_year"].dropna(), bins=np.arange(1989, 2028), color="steelblue", edgecolor="white")
ax.set_xlabel("Discovery year")
ax.set_ylabel("Planets announced")
plt.show()

## Save your work

File → Save a copy in Drive. Next session we'll load Gaia DR4 into this same notebook and see what lands in the red box.